<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/05_train_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Fine-tune YOLO (light, free T4)

A **light fine-tune of the COCO-pretrained `yolo11n.pt`** on our 20 classes, sized for the
free Colab T4. This is not training from scratch.

**Why:** the zero-training hybrid detector (notebook 08) misses key hazards. Its test-split
mAP50: curb 0.001, barrier 0.01, pothole 0.03, crosswalk 0.04, pole 0.08, manhole 0.08,
bollard 0.18, scooter 0.24, stairs 0.31. Only traffic_cone (0.77) was good.

```
reads   MyDrive/clear_vision/datasets/merged.zip + merged.done     (prepare_dataset, unpacked to /content)
        MyDrive/clear_vision/datasets/train_balanced.txt + data_balanced.yaml   (notebook 03)
        MyDrive/clear_vision/runs/compare_detectors/20261116_0617.json    (notebook 08 scores)
writes  MyDrive/clear_vision/runs/<RUN_NAME>/          weights (last, best, every 5th epoch), results.csv, plots
        MyDrive/clear_vision/runs/<RUN_NAME>_test/     test-split plots (confusion matrix, PR curves)
        MyDrive/clear_vision/exports/<RUN_NAME>/       best.pt, best.onnx, metrics.json, per_class_choice.yaml
```

| preset | model | imgsz | epochs | train subset | freeze | patience |
|---|---|---|---|---|---|---|
| **BUDGET** (default) | yolo11n.pt | 512 | 30 | 30% + every focus image | backbone (10 layers) | 10 |
| NORMAL | yolo11n.pt | 640 | 50 | 50% + every focus image | none | 15 |

- **Data:**
  - Training reads only the local copy in `/content`, never Drive.
  - The train list is notebook 03's balanced list, with rare classes repeated
    (`USE_BALANCED`). From it, a subset keeps **every image with a focus class** plus a
    `SUBSET_FRACTION` of the rest, with person-only images capped.
  - Val and test stay complete.
- **Time:** BUDGET aims at about 1-2 h on a T4. The time of epoch 1 and an estimate for the
  rest are printed.
- **Resumable:**
  - Weights are saved to Drive every epoch (`last.pt`), plus every 5th epoch.
  - After a disconnect, run Settings, sections 1-2, then the **Resume** cell. You lose
    at most the epoch in progress.
  - Only weights, results.csv and plots go to Drive, about 100-200 MB per run.
- **Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU). Run the cells in order,
  skipping **Resume** unless you were disconnected. Kaggle instructions are at the end.

In [ ]:
# Settings
PRESET = "BUDGET"  # "BUDGET" (default, ~1-2 h on a T4) or "NORMAL"
RUN_NAME = "budget_v1"  # one name per run; keep it to resume after a disconnect
USE_BALANCED = True  # train on notebook 03's balanced list; False = plain data.yaml (all train images once)
FOCUS_CLASSES = ["curb", "stairs", "pothole", "manhole", "bollard", "barrier", "traffic_cone", "pole",
                 "scooter", "crosswalk"]  # every train image with one of these is kept in the subset
PERSON_CAP = 2000  # at most this many person-only images in the subset
SEED = 42
EXPORT_ONNX = True  # also export best.onnx
MISS_CONF = 0.25  # confidence at which "missed" objects are judged (like the live app)
RECOMPUTE_ZERO_TRAINING = False  # True: re-score the hybrid on today's test split (slower, fairer if labels changed)

REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"  # Kaggle: see the last section
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
ZERO_TRAINING_JSON = f"{DRIVE_ROOT}/runs/compare_detectors/20261116_0617.json"
LOCAL_DATASET = "/content/datasets/merged"

# Preset values (edit here to override one, e.g. EPOCHS = 20)
_p = {
    "BUDGET": dict(MODEL="yolo11n.pt", IMGSZ=512, EPOCHS=30, SUBSET_FRACTION=0.3, FREEZE=10, BATCH=-1, PATIENCE=10),
    "NORMAL": dict(MODEL="yolo11n.pt", IMGSZ=640, EPOCHS=50, SUBSET_FRACTION=0.5, FREEZE=None, BATCH=-1, PATIENCE=15),
}[PRESET]
MODEL, IMGSZ, EPOCHS, SUBSET_FRACTION = _p["MODEL"], _p["IMGSZ"], _p["EPOCHS"], _p["SUBSET_FRACTION"]
FREEZE, BATCH, PATIENCE = _p["FREEZE"], _p["BATCH"], _p["PATIENCE"]
print(f"{PRESET}: {_p}")

## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages, GPU, free space
import json
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

ON_KAGGLE = Path("/kaggle").exists()
if not ON_KAGGLE:
    from google.colab import drive

    drive.mount("/content/drive")

REPO_DIR = Path("/kaggle/working/clear-vision" if ON_KAGGLE else "/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics

sys.path.insert(0, str(REPO_DIR))
import scripts.train_utils as tu  # noqa: E402
from src.colab_data import apply_balanced, prepare_dataset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> T4 GPU (CPU training would take days).")
print(f"GPU: {torch.cuda.get_device_name(0)}")

MERGED_ZIP = Path(MERGED_ZIP)
RUNS = Path(DRIVE_ROOT) / "runs"
EXPORTS = Path(DRIVE_ROOT) / "exports" / RUN_NAME
free = tu.check_space(RUNS, 500 * 10**6)  # ~100-200 MB per run, plus exports and margin
print(f"Drive free: {free / 1e9:.1f} GB")

## 2. Data

- `prepare_dataset()` copies `merged.zip` to `/content` and unzips it.
- `apply_balanced()` adds notebook 03's balanced list. If the list was made for an older
  `merged.zip`, this stops: re-run notebook 03, or set `USE_BALANCED = False`.
- Then it builds the train subset: every focus image (with its balanced repeats), plus
  `SUBSET_FRACTION` of the rest, with person-only images capped. The subset is the same
  every time (seeded), which resuming needs.

In [ ]:
# 2. Unpack, apply the balanced list, build the train subset
DATA_YAML = prepare_dataset(MERGED_ZIP, LOCAL_DATASET)
DATASET = DATA_YAML.parent
CLASSES = list(json.loads(MERGED_ZIP.with_name("merge_report.json").read_text())["classes"])

if USE_BALANCED:
    try:
        BASE_YAML = apply_balanced(MERGED_ZIP, DATASET)
    except RuntimeError as e:
        raise SystemExit(
            f"\n{e}\n\nThe balanced train list can't be used with the current merged.zip. Either re-run "
            "notebook 03 (it writes a new train_balanced.txt next to merged.zip), or set USE_BALANCED = False "
            "in Settings to train on the plain data.yaml."
        ) from None
else:
    BASE_YAML = DATA_YAML

lines = tu.train_lines(DATASET, balanced=USE_BALANCED)
SUBSET, SUBSET_INFO = tu.build_subset(lines, tu.train_image_classes(DATASET, len(CLASSES)), CLASSES,
                                      SUBSET_FRACTION, FOCUS_CLASSES, PERSON_CAP, SEED)
TRAIN_YAML = tu.write_subset(DATASET, BASE_YAML, SUBSET)
print(f"train list: {'balanced' if USE_BALANCED else 'plain'}, {SUBSET_INFO['source_lines']:,} lines "
      f"({SUBSET_INFO['source_images']:,} images)")
print(f"subset: {SUBSET_INFO['focus_images']:,} focus images (all kept) + {SUBSET_INFO['other_kept']:,} of "
      f"{SUBSET_INFO['other_images']:,} others ({SUBSET_FRACTION:.0%}, {SUBSET_INFO['person_only_dropped']:,} "
      f"person-only dropped) = {SUBSET_INFO['subset_images']:,} images, {SUBSET_INFO['subset_lines']:,} lines per epoch")
print(f"train with: {TRAIN_YAML}")

## 3. Train

- Fine-tunes `yolo11n.pt` (COCO-pretrained). With `FREEZE = 10` the backbone stays fixed
  and only the neck and head learn, which is faster and safer on little data.
- **Augmentations for walking footage:**
  - colour and brightness (hsv), ±5° rotation, ±50% scale
  - mosaic, switched off for the last 5 epochs
  - horizontal flips; vertical flips are off, since the world is never upside down
- After epoch 1 it prints how long an epoch takes and an estimate for the whole run.

In [ ]:
# 3. Train (if the run already exists, use the Resume cell below instead)
from ultralytics import YOLO

if tu.last_checkpoint(RUNS, RUN_NAME) is not None:
    raise SystemExit(f"{RUNS / RUN_NAME} already has weights. Run the Resume cell, or change RUN_NAME for a new run.")

TIMER = tu.EpochTimer()
model = YOLO(str(REPO_DIR / "models" / MODEL))  # downloads the COCO-pretrained weights once
model.add_callback("on_fit_epoch_end", TIMER)
model.train(
    data=str(TRAIN_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, freeze=FREEZE, patience=PATIENCE,
    project=str(RUNS), name=RUN_NAME, exist_ok=True, save_period=5, seed=SEED, plots=True,
    **tu.AUGMENT,
)

### Resume after a disconnect

Run Settings, sections 1 and 2 (they rebuild the same local dataset and subset), then this
cell. It continues from `last.pt` on Drive with the original settings.

In [ ]:
# 3b. Resume from last.pt
from ultralytics import YOLO

last = tu.last_checkpoint(RUNS, RUN_NAME)
if last is None:
    raise SystemExit(f"No {RUNS / RUN_NAME}/weights/last.pt: nothing to resume. Run section 3.")
TIMER = tu.EpochTimer()
model = YOLO(str(last))
model.add_callback("on_fit_epoch_end", TIMER)
model.train(resume=True)

## 4. Test split: fine-tuned vs zero-training

- **Ultralytics `val(split="test")`** gives precision, recall and the confusion matrix.
- **Same metric as notebook 08:** for the comparison table, the fine-tuned model is scored
  with notebook 08's own evaluator (COCO-style AP at IoU 0.5, threshold 0.01). Both columns
  then mean the same thing.
- If the test labels changed since notebook 08 (for example, pseudo-labels from notebook
  09), set `RECOMPUTE_ZERO_TRAINING = True` to re-score the hybrid on today's labels.

In [ ]:
# 4. Validate on TEST and compare with notebook 08
import pandas as pd

import scripts.compare_detectors as cd
from src.detector import Detector

BEST = RUNS / RUN_NAME / "weights" / "best.pt"
best_model = YOLO(str(BEST))
TEST = best_model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=16, conf=0.001, plots=True,
                      project=str(RUNS), name=f"{RUN_NAME}_test", exist_ok=True)
ULTRA = tu.per_class_from_ultralytics(TEST, CLASSES)

pairs = cd.load_split(DATA_YAML, "test")
detector = Detector(BEST, conf=0.01, imgsz=IMGSZ)
PREDS, GTS = tu.predict_split(detector, pairs, CLASSES)
FINETUNED = cd.ap50_per_class(PREDS, GTS, CLASSES)

if RECOMPUTE_ZERO_TRAINING:
    import yaml

    det_cfg = yaml.safe_load((REPO_DIR / "configs" / "default.yaml").read_text())["detector"]
    ZERO = cd.run_eval(det_cfg, ["hybrid"], DATA_YAML, CLASSES, 0.01, None)["hybrid"]
elif Path(ZERO_TRAINING_JSON).exists():
    ZERO = tu.load_zero_training(Path(ZERO_TRAINING_JSON))
else:
    print(f"{ZERO_TRAINING_JSON} not found: the zero-training column stays empty.")
    ZERO = {}

ROWS = tu.comparison_rows(CLASSES, ZERO, FINETUNED, {c: v["recall"] for c, v in ULTRA.items()})
print(f"Test split: mAP50 {TEST.box.map50:.3f}, mAP50-95 {TEST.box.map:.3f} (Ultralytics)\n")
print(tu.format_comparison(ROWS))

table = pd.DataFrame(ROWS).set_index("class").drop(columns="critical")
display(table.style.format(lambda v: "-" if v is None or v != v else f"{v:.3f}", subset=["zero-training hybrid", "fine-tuned", "recall"])
        .apply(lambda row: ["background-color: #fff3cd; color: #000; font-weight: bold" if row.name in tu.CRITICAL_CLASSES
                            and col == "recall" else "" for col in row.index], axis=1)
        .apply(lambda row: ["color: #0a7a2f" if v == "yes" else "color: #b42318" if v == "no" else "" for v in row], subset=["better?"])
        .set_caption("mAP50 on the TEST split; recall highlighted for the critical classes"))

In [ ]:
# 4b. Confusion matrix (normalized: each column = one true class)
from IPython.display import Image

cm = Path(TEST.save_dir) / "confusion_matrix_normalized.png"
display(Image(filename=str(cm), width=900))

## 5. Per-class choice for the app

For each class, `per_class_choice.yaml` names the model that scored higher on the test
split: `finetuned`, or `openvocab` (the zero-training path). For COCO classes, the hybrid's
zero-training score comes from yolo11n; `zero_training_model` records which model it was.
Ties and classes without a score keep the fine-tuned model.

In [ ]:
# 5. Write per_class_choice.yaml to exports/<RUN_NAME>/
import yaml

EXPORTS.mkdir(parents=True, exist_ok=True)
CHOICE = tu.per_class_choice(CLASSES, ZERO, FINETUNED, coco_classes=tu.COCO_CLASSES)
choice_doc = {"run": RUN_NAME, "metric": "mAP50 on the test split (notebook 08 evaluator)", **CHOICE}
(EXPORTS / "per_class_choice.yaml").write_text(
    "# Written by notebooks/05_train_yolo.ipynb: which model to trust per class in the app's hybrid mode.\n"
    + yaml.safe_dump(choice_doc, sort_keys=False))
print(yaml.safe_dump(CHOICE["choice"], sort_keys=False))
print(f"wrote {EXPORTS / 'per_class_choice.yaml'}")

## 6. Missed critical objects

These are test-split objects of the critical classes (curb, stairs, pothole, manhole, pole)
that the fine-tuned model did **not** find at `MISS_CONF`, the threshold the live app works
near.
- **Red:** the missed object.
- **Gray:** what the model predicted instead.
- **"seen as X":** it found the object but called it another class.

In [ ]:
# 6. 12 missed critical objects (false negatives)
import cv2
import matplotlib.pyplot as plt

MISSES = tu.find_misses(PREDS, GTS, tu.CRITICAL_CLASSES, conf=MISS_CONF)
totals = {c: sum(1 for boxes in GTS.values() for b in boxes if b[0] == c) for c in tu.CRITICAL_CLASSES}
missed = tu.group_misses(MISSES)
for c in tu.CRITICAL_CLASSES:
    print(f"{c:8s} missed {missed.get(c, 0):5d} of {totals[c]:5d} test objects at conf {MISS_CONF}")
grid = tu.miss_grid(MISSES, PREDS, n=12, seed=SEED, conf=MISS_CONF)
if grid is None:
    print("No missed critical objects.")
else:
    plt.figure(figsize=(18, 10))
    plt.imshow(cv2.cvtColor(grid, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 7. Export

In [ ]:
# 7. best.pt (+ best.onnx) and metrics.json to exports/<RUN_NAME>/
import csv

shutil.copyfile(BEST, EXPORTS / "best.pt")
if EXPORT_ONNX:
    onnx = Path(best_model.export(format="onnx", imgsz=IMGSZ, simplify=True))
    shutil.copyfile(onnx, EXPORTS / "best.onnx")

results_csv = RUNS / RUN_NAME / "results.csv"
epochs_run = sum(1 for _ in csv.DictReader(results_csv.open())) if results_csv.exists() else None
metrics = {
    "run": RUN_NAME,
    "preset": PRESET,
    "commit": commit.strip(),
    "settings": {"model": MODEL, "imgsz": IMGSZ, "epochs": EPOCHS, "epochs_run": epochs_run, "freeze": FREEZE,
                 "batch": BATCH, "patience": PATIENCE, "subset_fraction": SUBSET_FRACTION, "person_cap": PERSON_CAP,
                 "use_balanced": USE_BALANCED, "focus_classes": FOCUS_CLASSES, "augment": tu.AUGMENT, "seed": SEED},
    "data": {"subset": SUBSET_INFO, "merged_done": json.loads(MERGED_ZIP.with_suffix(".done").read_text())},
    "epoch_time": TIMER.messages if "TIMER" in globals() else [],
    "test": {"map50": float(TEST.box.map50), "map50_95": float(TEST.box.map), "precision": float(TEST.box.mp),
             "recall": float(TEST.box.mr), "per_class": ULTRA},
    "comparison": {"metric": "mAP50, notebook 08 evaluator", "zero_training_source": "recomputed" if RECOMPUTE_ZERO_TRAINING
                   else ZERO_TRAINING_JSON, "rows": ROWS},
    "missed_critical": {"conf": MISS_CONF, "missed": missed, "total": totals},
}
(EXPORTS / "metrics.json").write_text(json.dumps(metrics, indent=2, default=lambda v: None))
for p in sorted(EXPORTS.iterdir()):
    print(f"{p.name:24s} {p.stat().st_size / 1e6:7.2f} MB")
run_mb = sum(p.stat().st_size for p in (RUNS / RUN_NAME).rglob("*") if p.is_file()) / 1e6
print(f"run folder on Drive: {run_mb:.0f} MB")

## Running on Kaggle instead (free weekly GPU)

If Colab's free GPU time runs out, Kaggle gives about 30 GPU hours a week (T4 x2 or P100)
and runs this notebook with a few changes.

1. **Upload the data once as a private Kaggle Dataset** (kaggle.com → Datasets → New), named
   for example `clear-vision-merged`. Download these files from
   `MyDrive/clear_vision/datasets/` and upload them together:
   - `merged.zip` and `merged.done`
   - `merge_report.json`
   - `train_balanced.txt` and `data_balanced.yaml`

   Optionally also add `compare_detectors/20261116_0617.json`.
2. **New Notebook** → File → Import Notebook → GitHub → `Zikirullo/clear-vision`,
   `notebooks/05_train_yolo.ipynb`. In the right panel:
   - Accelerator: **GPU T4 x2** (or P100)
   - Internet: **on**
   - Add Input: your dataset
3. **In Settings, change the paths.** Kaggle has no Drive; `/kaggle/input` is read-only, and
   `/kaggle/working` is kept as the notebook's output (20 GB).
   ```python
   DRIVE_ROOT = "/kaggle/working/clear_vision"                       # runs/ and exports/ go here
   MERGED_ZIP = "/kaggle/input/clear-vision-merged/merged.zip"        # merged.done etc. next to it
   ZERO_TRAINING_JSON = "/kaggle/input/clear-vision-merged/20261116_0617.json"
   LOCAL_DATASET = "/tmp/datasets/merged"                             # scratch disk, not saved
   ```
   The setup cell detects Kaggle and skips the Drive mount.
4. **Resuming:**
   - A Kaggle session lasts up to 12 h, which is enough for BUDGET in one go.
   - If it stops, the files in `/kaggle/working` are saved as the notebook's output. Add
     that output as an input to a new session, copy `runs/<RUN_NAME>` back into
     `/kaggle/working/clear_vision/runs/`, and run the Resume cell.
5. **Afterwards:** download `exports/<RUN_NAME>/` from the Output tab and put it in
   `MyDrive/clear_vision/exports/`.